# 02 — Evaluation Setup

This notebook defines, and then uses for every model/hyperparameter/
representation/routing-policy choice, the full evaluation pipeline: the
temporal train/validation/test split, the candidate/exclusion policy, the
metrics, and a validation-only search over hyperparameters and interaction
representations.

**Test-set discipline (applies to the whole project, not just this
notebook):** the test set is never used for model, hyperparameter,
representation, or routing-policy selection. It is used only for final
evaluation and post-hoc diagnostics, in `03_models.ipynb` and
`04_product_analysis.ipynb`. Every selection decision in this notebook,
including the relevance-threshold sensitivity check in Section 6, is made
on validation only.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

from src import data as D
from src import evaluation as E
from src import metrics as M
from src.recommenders import (
    PopularityRecommender, ItemKNNRecommender, ContentBasedRecommender,
    FactorModelRecommender, HybridRecommender,
)

K = 10
pd.set_option("display.max_columns", 50)

## 1. Load data and build the temporal split

For each user, interactions are sorted by timestamp and cut into three
contiguous blocks: the earliest ~70% go to **train**, the next ~15% to
**validation**, and the most recent ~15% to **test**. Every user contributes
to all three blocks because MovieLens-1M guarantees at least 20 ratings per
user (`01_data_and_eda.ipynb`).

- **train** — fit every model.
- **validation** — choose hyperparameters, the interaction representation,
  the content-history threshold, the hybrid weight, and (in
  `04_product_analysis.ipynb`) the routing policy.
- **test** — used only once, for final evaluation and post-hoc diagnostics
  in `03_models.ipynb` and `04_product_analysis.ipynb`.

In [2]:
users, movies, ratings = D.load_raw("../data")
ratings, user_id_to_idx, item_id_to_idx = D.add_index_columns(ratings)
df = D.temporal_split(ratings)
n_users = df["user_idx"].nunique()

print(df["split"].value_counts())
print((df["split"].value_counts(normalize=True) * 100).round(1))

split
train    700022
test     150177
val      150010
Name: count, dtype: int64
split
train    70.0
test     15.0
val      15.0
Name: proportion, dtype: float64


## 2. Candidate universe, exclusion rule, and recommendable truth

Every model ranks the same candidate set: **items observed at least once in
the training data available at that point** (train only, while tuning on
validation; train+validation, for the final test evaluation in `03`). For
every user, all items they have *already interacted with* (any rating, not
just relevant ones) are removed from their candidate list before ranking.

**A relevant item that is not in the candidate catalog cannot be
recommended by any model — this is item cold start, not a ranking
failure.** `D.get_truth()` returns every relevant (`rating >= 4`) item in a
split, regardless of whether it is in the candidate catalog yet.
`D.restrict_truth_to_catalog()` intersects that truth with the catalog to
produce the **recommendable truth** actually used for Recall/Precision/
MAP/NDCG below — and reports what got excluded, so the item-cold-start rate
is visible rather than silently folded into every model's score.

In [3]:
catalog_items = D.catalog_from_split(df, ["train"])
item_pos = E.item_position_map(catalog_items)
train_df = df[df["split"] == "train"]
train_df = train_df[train_df["item_idx"].isin(item_pos)]

seen = D.get_seen(df, ["train"])
raw_truth = D.get_truth(df, "val", threshold=D.RELEVANT_RATING_THRESHOLD)
truth, truth_stats = D.restrict_truth_to_catalog(raw_truth, item_pos)
pop_count = E.item_pop_count(df, ["train"])
feature_matrix, genre_names = D.build_movie_features(movies, catalog_items, item_id_to_idx)

users_to_eval = list(truth.keys())
print(f"Candidate catalog: {len(catalog_items)} movies")
print(f"Validation relevant interactions (raw): {truth_stats['total_relevant_interactions']}")
print(f"  excluded (item not yet in candidate catalog): {truth_stats['excluded_not_in_catalog']} "
      f"({truth_stats['excluded_share']:.2%}) -- item cold start, see Limitations")
print(f"Evaluation cohort: {truth_stats['users_after']} / {n_users} users "
      f"({truth_stats['users_after']/n_users:.1%}) have >= 1 RECOMMENDABLE relevant item in validation "
      f"(before the catalog restriction: {truth_stats['users_before']} users)")

Candidate catalog: 3649 movies
Validation relevant interactions (raw): 81075
  excluded (item not yet in candidate catalog): 12 (0.01%) -- item cold start, see Limitations
Evaluation cohort: 5920 / 6040 users (98.0%) have >= 1 RECOMMENDABLE relevant item in validation (before the catalog restriction: 5920 users)


The item-cold-start exclusion rate above is small but real (see the printed
percentage). It is reported here, once, as the reference number; the same
restriction is applied identically on the test set in `03_models.ipynb`,
where its own rate is reported again (the rates differ slightly because the
train and train+val catalogs differ in size).

The remaining cohort gap versus the total user count is the *other* reason
users can be missing from an evaluation cohort: a user who had zero
`rating >= 4` interactions in their validation slice at all (nothing to
restrict away — they simply produced no relevant item to evaluate against).
Both reasons are now reported separately rather than being blended into a
single unexplained "cohort size" number.

## 3. Metrics

All ranking metrics are computed at **K = 10**, against the recommendable
truth defined above.

- **Recall@10** — share of a user's relevant, recommendable items that appear in the top 10.
- **Precision@10** — share of the top 10 that are relevant.
- **MAP@10** — average precision at each rank a relevant item is hit, rewarding relevant items ranked higher.
- **NDCG@10** — like MAP, but discounts by `1/log2(rank+1)` and normalizes against the best possible ordering; the primary metric used for every selection decision in this notebook because it directly rewards ranking quality rather than just retrieval.

Beyond-accuracy metrics (computed in `03_models.ipynb` on the final test predictions):

- **Catalog Coverage@10** — share of the whole catalog that appears in *any* user's top 10, across all users. Low coverage means the system effectively only ever recommends a small subset of the catalog.
- **Novelty@10** — mean self-information `-log2(popularity share)` of recommended items; higher means less-popular ("long tail") items are being surfaced.
- **Intra-list Diversity@10** — mean pairwise `(1 - cosine similarity)` between the genre/year vectors of items *within a single user's* list; higher means a single list spans more different kinds of movies.
- **Average Recommended Popularity@10** — mean training-set interaction count of recommended items, a simple, literal popularity-bias indicator.

Implementations live in `src/metrics.py`.

## 4. Model selection on validation

Every model below is fit on **train only** and scored against the
**validation** recommendable truth, using the shared candidate/exclusion
policy from Section 2. The hyperparameter (or design choice) with the
highest validation NDCG@10 is kept; the final choices are collected into
`BEST_PARAMS` in Section 5, matching `src/pipeline.py::BEST_PARAMS`, and
reused without modification on the test set in `03_models.ipynb`.

### 4.1 Popularity baseline (no hyperparameters)

In [4]:
pop_model = PopularityRecommender().fit(catalog_items, pop_count)
rec_pop = E.generate_rec_lists(pop_model, users_to_eval, seen, catalog_items, item_pos, K)
print(M.evaluate_recommendations(rec_pop, truth, K))

{'Recall@K': 0.05103147769449089, 'Precision@K': 0.06363175675675677, 'MAP@K': 0.03551204512867459, 'NDCG@K': 0.07590711249503716, 'n_users': 5920}


### 4.2 Item-kNN (representation A: all interactions)

In [5]:
knn_model = ItemKNNRecommender().fit(train_df, catalog_items, item_pos)
rec_knn = E.generate_rec_lists(knn_model, users_to_eval, seen, catalog_items, item_pos, K)
print(M.evaluate_recommendations(rec_knn, truth, K))

{'Recall@K': 0.09302184328662427, 'Precision@K': 0.0860641891891892, 'MAP@K': 0.05466243390499344, 'NDCG@K': 0.11135307109847624, 'n_users': 5920}


### 4.3 Content-based: choosing the history threshold

The content-based user profile is built from movies in a user's *train*
history rated at or above `history_min_rating`. Using every interaction
(including disliked movies) to build the taste profile is questionable in
the same way as the relevance definition itself — so this threshold is
tuned on validation rather than assumed.

In [6]:
cb_results = {}
for min_rating in [None, 3, 4]:
    cb_model = ContentBasedRecommender().fit(
        train_df, catalog_items, item_pos, feature_matrix, min_rating, pop_model.scores
    )
    rec_cb = E.generate_rec_lists(cb_model, users_to_eval, seen, catalog_items, item_pos, K)
    m = M.evaluate_recommendations(rec_cb, truth, K)
    cb_results[min_rating] = (m, cb_model)
    print(f"min_rating={min_rating}: {m}")

best_min_rating = max(cb_results, key=lambda mr: cb_results[mr][0]['NDCG@K'])
best_cb_model = cb_results[best_min_rating][1]
print(f"\nChosen history_min_rating = {best_min_rating}")

min_rating=None: {'Recall@K': 0.015673257720260944, 'Precision@K': 0.012449324324324326, 'MAP@K': 0.0070656658465438225, 'NDCG@K': 0.01670002419054012, 'n_users': 5920}


min_rating=3: {'Recall@K': 0.016315461916201177, 'Precision@K': 0.012736486486486488, 'MAP@K': 0.007021946797988465, 'NDCG@K': 0.01699487589758238, 'n_users': 5920}


min_rating=4: {'Recall@K': 0.01578709859115193, 'Precision@K': 0.012820945945945947, 'MAP@K': 0.007095751071011488, 'NDCG@K': 0.017195966460638753, 'n_users': 5920}

Chosen history_min_rating = 4


### 4.4 ALS — and the orientation bug found during the audit

**Finding from the audit of the previous version of this project:** the ALS
model was fit as `als_model.fit(train_matrix)` where `train_matrix` had
shape `(n_items, n_users)`. The `implicit` library's `AlternatingLeastSquares.fit`
expects a `(n_users, n_items)` matrix — the matrix was transposed. This
silently swaps the meaning of "user factors" and "item factors": row `u` of
`user_factors` ends up describing *item* `u`, not user `u`. Every downstream
recommendation was therefore scored against the wrong entity, which is the
real reason the old notebook's implicit-ALS evaluation collapsed to exactly
zero — not evidence that "ALS doesn't work for temporal splits".

The cell below reproduces the bug in isolation, on this project's own data
and metrics, to confirm the diagnosis before fixing it.

In [7]:
# --- Reproduce the orientation bug for the audit record ---
from scipy.sparse import csr_matrix

wrong_rows = np.array([item_pos[i] for i in train_df["item_idx"].values])
wrong_cols = train_df["user_idx"].values
wrong_vals = 1.0 + 2.0 * (train_df["rating"].values.astype(float) - 1.0)
wrong_matrix = csr_matrix((wrong_vals, (wrong_rows, wrong_cols)), shape=(len(catalog_items), n_users))

buggy_model = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=20, random_state=42)
buggy_model.fit(wrong_matrix, show_progress=False)  # fit on (items, users) -- WRONG orientation

print("fit() was called on a (n_catalog, n_users) matrix instead of (n_users, n_catalog).")
print(f"  wrong_matrix.shape = {wrong_matrix.shape}  (rows are treated as 'users' by the library)")
print(f"  user_factors.shape = {buggy_model.user_factors.shape}  <- indexed by CATALOG POSITION, not user_idx")
print(f"  item_factors.shape = {buggy_model.item_factors.shape}  <- indexed by USER_IDX, not catalog position")
print()
print("recommend(userid=user_idx, ...) would look up user_factors[user_idx] -- "
      "but that row describes catalog item `user_idx`, not user `user_idx`. "
      "Because n_users (6040) > n_catalog (3649), most indices stay in-bounds, "
      "so the call does not raise an error: it silently returns a ranking with "
      "no relationship to the target user, which is why the original "
      "notebook's implicit-ALS metrics came out at (or near) zero.")

fit() was called on a (n_catalog, n_users) matrix instead of (n_users, n_catalog).
  wrong_matrix.shape = (3649, 6040)  (rows are treated as 'users' by the library)
  user_factors.shape = (3649, 64)  <- indexed by CATALOG POSITION, not user_idx
  item_factors.shape = (6040, 64)  <- indexed by USER_IDX, not catalog position

recommend(userid=user_idx, ...) would look up user_factors[user_idx] -- but that row describes catalog item `user_idx`, not user `user_idx`. Because n_users (6040) > n_catalog (3649), most indices stay in-bounds, so the call does not raise an error: it silently returns a ranking with no relationship to the target user, which is why the original notebook's implicit-ALS metrics came out at (or near) zero.


In [8]:
# --- Correct orientation: (n_users, n_catalog), confidence-weighted, representation A ---
conf_matrix_a = E.build_confidence_matrix(df, ["train"], n_users, catalog_items, item_pos, alpha=2.0)

als_results = {}
for factors in [32, 64, 128]:
    for reg in [0.01, 0.1]:
        model = AlternatingLeastSquares(factors=factors, regularization=reg, iterations=20, random_state=42)
        als = FactorModelRecommender(model).fit(conf_matrix_a)
        rec_als = E.generate_rec_lists(als, users_to_eval, seen, catalog_items, item_pos, K)
        m = M.evaluate_recommendations(rec_als, truth, K)
        als_results[(factors, reg)] = (m, als)
        print(f"factors={factors} reg={reg}: {m}")

best_als_params = max(als_results, key=lambda p: als_results[p][0]['NDCG@K'])
best_als_model = als_results[best_als_params][1]
print(f"\nChosen (factors, regularization) = {best_als_params}")

factors=32 reg=0.01: {'Recall@K': 0.08649014769157058, 'Precision@K': 0.06503378378378379, 'MAP@K': 0.041315762935554605, 'NDCG@K': 0.08805174792480282, 'n_users': 5920}


factors=32 reg=0.1: {'Recall@K': 0.08602169035818213, 'Precision@K': 0.06483108108108109, 'MAP@K': 0.04111784576349308, 'NDCG@K': 0.08783156413879285, 'n_users': 5920}


factors=64 reg=0.01: {'Recall@K': 0.08718553589704399, 'Precision@K': 0.06339527027027027, 'MAP@K': 0.04024347386289351, 'NDCG@K': 0.08664407579406526, 'n_users': 5920}


factors=64 reg=0.1: {'Recall@K': 0.08692193463087526, 'Precision@K': 0.06314189189189189, 'MAP@K': 0.04001221765451676, 'NDCG@K': 0.08628614315949143, 'n_users': 5920}


factors=128 reg=0.01: {'Recall@K': 0.08795810966183075, 'Precision@K': 0.06108108108108108, 'MAP@K': 0.040288525079075675, 'NDCG@K': 0.08607666031358473, 'n_users': 5920}


factors=128 reg=0.1: {'Recall@K': 0.08819877828959298, 'Precision@K': 0.06123310810810811, 'MAP@K': 0.04029139691881507, 'NDCG@K': 0.08611044002882946, 'n_users': 5920}

Chosen (factors, regularization) = (32, 0.01)


Correcting the orientation alone takes NDCG@10 from this silently-broken ~0
up to roughly **0.09** — comparable to Item-kNN. ALS is a legitimate,
competitive model on this data once the bug is fixed; the earlier "ALS
doesn't suit a temporal split" conclusion in the previous version of this
project was an artifact of a matrix-shape bug, not a property of the
algorithm. This is exactly the kind of mistake the audit was asked to catch
rather than take at face value.

### 4.5 A ranking-loss factorization model in place of LightFM

The brief calls for a hybrid built with **LightFM** (collaborative +
genre item features, WARP/BPR loss). LightFM's PyPI source distribution
does not build on this environment (Windows, Python 3.12): its `setup.py`
predates modern `setuptools` and fails at the metadata-generation step
(`AttributeError: 'dict' object has no attribute '__LIGHTFM_SETUP__'`)
regardless of build-isolation settings, and the project has not published a
compatible release since 2020. Rather than lose time fighting an unmaintained
build toolchain, we substitute **Bayesian Personalized Ranking (BPR)** from
the `implicit` library already used for ALS: it optimizes a pairwise ranking
loss (the same family LightFM's WARP/BPR losses belong to) directly from
implicit feedback, which is the property we actually wanted from LightFM.
It does not consume genre features directly — that gap is covered by
hybridizing it with the content-based model in Section 4.6.

In [9]:
bpr_results = {}
for factors in [32, 64, 128]:
    model = BayesianPersonalizedRanking(factors=factors, iterations=100, random_state=42)
    bpr = FactorModelRecommender(model).fit(conf_matrix_a)
    rec_bpr = E.generate_rec_lists(bpr, users_to_eval, seen, catalog_items, item_pos, K)
    m = M.evaluate_recommendations(rec_bpr, truth, K)
    bpr_results[factors] = (m, bpr)
    print(f"factors={factors}: {m}")

best_bpr_factors = max(bpr_results, key=lambda f: bpr_results[f][0]['NDCG@K'])
best_bpr_model = bpr_results[best_bpr_factors][1]
print(f"\nChosen BPR factors = {best_bpr_factors}")

factors=32: {'Recall@K': 0.06383838711663412, 'Precision@K': 0.04442567567567567, 'MAP@K': 0.03022787429223441, 'NDCG@K': 0.06438194842456162, 'n_users': 5920}


factors=64: {'Recall@K': 0.06365270693013862, 'Precision@K': 0.043733108108108106, 'MAP@K': 0.029838589998374222, 'NDCG@K': 0.0635019551491762, 'n_users': 5920}


factors=128: {'Recall@K': 0.06522709200965261, 'Precision@K': 0.0441722972972973, 'MAP@K': 0.03131704850045624, 'NDCG@K': 0.06485402752667359, 'n_users': 5920}

Chosen BPR factors = 128


### 4.6 Hybrid: which collaborative signal pairs best with content?

The hybrid combines the content-based score with one collaborative model's
score (each min-max normalized per user, then blended by weight `alpha`).
Rather than assume a partner and a 50/50 weight, both the partner model and
`alpha` are chosen on validation.

In [10]:
hybrid_search = {}
for partner_name, partner_model in [("BPR", best_bpr_model), ("ALS", best_als_model), ("Item-kNN", knn_model)]:
    for alpha in np.round(np.arange(0.0, 1.01, 0.1), 2):
        hyb = HybridRecommender(best_cb_model, partner_model, alpha)
        rec_hyb = E.generate_rec_lists(hyb, users_to_eval, seen, catalog_items, item_pos, K)
        m = M.evaluate_recommendations(rec_hyb, truth, K)
        hybrid_search[(partner_name, alpha)] = m['NDCG@K']

best_partner, best_alpha = max(hybrid_search, key=hybrid_search.get)
print(f"Best pairing: content-based + {best_partner}, alpha={best_alpha} "
      f"(NDCG@10={hybrid_search[(best_partner, best_alpha)]:.5f})")

pd.Series(hybrid_search).unstack(0).round(4)

Best pairing: content-based + Item-kNN, alpha=0.1 (NDCG@10=0.11170)


,ALS,BPR,Item-kNN
0.0,0.0881,0.0649,0.1114
0.1,0.0894,0.0659,0.1117
0.2,0.0913,0.0658,0.1082
0.3,0.0914,0.0651,0.1026
0.4,0.0899,0.0628,0.0966
0.5,0.0871,0.0586,0.0902
0.6,0.0826,0.0539,0.0826
0.7,0.0754,0.0483,0.0725
0.8,0.0640,0.0430,0.0619
0.9,0.0495,0.0352,0.0488


`alpha` here is the content-based model's weight: `alpha=0.0` is pure
collaborative, `alpha=1.0` is pure content-based. The best pairing is a
**small** content-based contribution (`alpha≈0.1`) added to the strongest
collaborative model on this data — content alone is too weak to carry the
combination, but a modest blend nudges the collaborative model's ranking.

### 4.7 Explicit-rating semantics: does treating every interaction as positive actually help?

Item-kNN, ALS and BPR all currently learn from **every** observed
interaction, including `rating = 1`, as an implicit positive signal (with
low confidence weight, but still positive) — the dataset is explicit
ratings, so this is a modeling assumption, not a fact about the data. This
section compares that against a **positive-only** representation that
removes disliked/neutral interactions from the training signal entirely,
using each model's already-chosen hyperparameters from 4.2/4.4/4.5 so the
*only* thing that changes is which interactions the model is allowed to
learn from:

- **A (all interactions, current default):** every train interaction is
  used, exactly as in 4.2/4.4/4.5.
- **B (positive-only):** only interactions with `rating >= 4` (the same
  threshold used for evaluation relevance) are used. For Item-kNN this
  means the item-item similarity matrix and a user's scoring history are
  both built only from their liked movies. For ALS/BPR, the confidence
  matrix is built only from relevant interactions.

The representation with the higher **validation** NDCG@10 is kept per
model; test is not consulted.

In [11]:
representation_results = {}

# --- Item-kNN ---
knn_pos_df = train_df[train_df["rating"] >= D.RELEVANT_RATING_THRESHOLD]
knn_b_model = ItemKNNRecommender().fit(knn_pos_df, catalog_items, item_pos)
rec_knn_b = E.generate_rec_lists(knn_b_model, users_to_eval, seen, catalog_items, item_pos, K)
m_knn_a = M.evaluate_recommendations(rec_knn, truth, K)
m_knn_b = M.evaluate_recommendations(rec_knn_b, truth, K)
representation_results["Item-kNN"] = {"A_all": m_knn_a["NDCG@K"], "B_positive_only": m_knn_b["NDCG@K"]}

# --- ALS (reuse the hyperparameters chosen in 4.4) ---
als_factors, als_reg = best_als_params
conf_matrix_b = E.build_confidence_matrix(
    df, ["train"], n_users, catalog_items, item_pos, alpha=2.0, min_rating=D.RELEVANT_RATING_THRESHOLD
)
als_b_model = FactorModelRecommender(
    AlternatingLeastSquares(factors=als_factors, regularization=als_reg, iterations=20, random_state=42)
).fit(conf_matrix_b)
rec_als_a = E.generate_rec_lists(best_als_model, users_to_eval, seen, catalog_items, item_pos, K)
rec_als_b = E.generate_rec_lists(als_b_model, users_to_eval, seen, catalog_items, item_pos, K)
m_als_a = M.evaluate_recommendations(rec_als_a, truth, K)
m_als_b = M.evaluate_recommendations(rec_als_b, truth, K)
representation_results["ALS"] = {"A_all": m_als_a["NDCG@K"], "B_positive_only": m_als_b["NDCG@K"]}

# --- BPR (reuse the hyperparameters chosen in 4.5) ---
bpr_b_model = FactorModelRecommender(
    BayesianPersonalizedRanking(factors=best_bpr_factors, iterations=100, random_state=42)
).fit(conf_matrix_b)
rec_bpr_a = E.generate_rec_lists(best_bpr_model, users_to_eval, seen, catalog_items, item_pos, K)
rec_bpr_b = E.generate_rec_lists(bpr_b_model, users_to_eval, seen, catalog_items, item_pos, K)
m_bpr_a = M.evaluate_recommendations(rec_bpr_a, truth, K)
m_bpr_b = M.evaluate_recommendations(rec_bpr_b, truth, K)
representation_results["BPR"] = {"A_all": m_bpr_a["NDCG@K"], "B_positive_only": m_bpr_b["NDCG@K"]}

pd.DataFrame(representation_results).T.round(5)

,A_all,B_positive_only
Item-kNN,0.11135,0.10827
ALS,0.08805,0.09984
BPR,0.06485,0.07779


In [12]:
# Choose per model, validation NDCG@10 only; None = representation A, threshold = representation B
knn_representation = None if m_knn_a["NDCG@K"] >= m_knn_b["NDCG@K"] else D.RELEVANT_RATING_THRESHOLD
als_representation = None if m_als_a["NDCG@K"] >= m_als_b["NDCG@K"] else D.RELEVANT_RATING_THRESHOLD
bpr_representation = None if m_bpr_a["NDCG@K"] >= m_bpr_b["NDCG@K"] else D.RELEVANT_RATING_THRESHOLD

final_knn_model = knn_model if knn_representation is None else knn_b_model
final_als_model = best_als_model if als_representation is None else als_b_model
final_bpr_model = best_bpr_model if bpr_representation is None else bpr_b_model

print("Chosen representation per model (None = A/all interactions, 4 = B/positive-only):")
print(f"  Item-kNN: {knn_representation}   ALS: {als_representation}   BPR: {bpr_representation}")

Chosen representation per model (None = A/all interactions, 4 = B/positive-only):
  Item-kNN: None   ALS: 4   BPR: 4


**Result: the two model families disagree, and both choices are kept.**

- **ALS and BPR clearly prefer the positive-only representation (B).**
  ALS improves from NDCG@10 = 0.0881 (A) to **0.0998** (B), BPR from 0.0651
  to **0.0778** — both are proper implicit-feedback factorization models,
  and feeding them a `rating=1` interaction as "the user consumed this,
  confidence=1" is a real semantic mismatch: it is not distinguishable from
  "we barely have any confidence either way", when what it actually means
  is "the user disliked this". Removing those rows measurably helps both.
- **Item-kNN slightly prefers the all-interactions representation (A)**
  (0.1114 vs. 0.1083 for B). Item-kNN already weights its similarity
  contributions by rating value (Section "Models Compared" in the README),
  so a `rating=1` interaction contributes a small positive weight rather
  than being treated as a strong positive — the extra co-occurrence data
  from keeping all interactions outweighs the noise from that residual
  mis-weighting.

So the chosen representation is **model-specific, not a single global
choice**: Item-kNN keeps representation A, ALS and BPR switch to
representation B. This changes `src/pipeline.py::BEST_PARAMS` and is
carried through to the final test evaluation in `03_models.ipynb`.

## 5. Chosen hyperparameters and representations

In [13]:
BEST_PARAMS = {
    "content_min_rating": best_min_rating,
    "als_factors": best_als_params[0],
    "als_regularization": best_als_params[1],
    "bpr_factors": best_bpr_factors,
    "hybrid_partner": best_partner,
    "hybrid_alpha": best_alpha,
    "itemknn_min_rating": knn_representation,
    "als_min_rating": als_representation,
    "bpr_min_rating": bpr_representation,
}
BEST_PARAMS

{'content_min_rating': 4,
 'als_factors': 32,
 'als_regularization': 0.01,
 'bpr_factors': 128,
 'hybrid_partner': 'Item-kNN',
 'hybrid_alpha': np.float64(0.1),
 'itemknn_min_rating': None,
 'als_min_rating': 4,
 'bpr_min_rating': 4}

These values match `src/pipeline.py::BEST_PARAMS`, used as-is (no further
tuning, on any split) in `03_models.ipynb` to fit final models on
train+validation and evaluate on test.

## 6. Sensitivity check: does the relevance threshold change which model wins?

This is a **validation-only** robustness check, not a re-selection: it asks
whether picking `rating >= 3` instead of `rating >= 4` as the relevance
threshold (Section 2, and `01_data_and_eda.ipynb`) would have changed which
model looks best, using the models already finalized above. It is run here,
on validation, precisely so that it can never influence the test numbers.

In [14]:
partner_model_map = {"Item-kNN": final_knn_model, "ALS": final_als_model, "BPR": final_bpr_model}
final_hybrid_model = HybridRecommender(best_cb_model, partner_model_map[best_partner], best_alpha)

final_models = {
    "Popularity": pop_model,
    "Item-kNN": final_knn_model,
    "Content-Based": best_cb_model,
    "ALS": final_als_model,
    "BPR": final_bpr_model,
    "Hybrid": final_hybrid_model,
}

raw_truth_r3 = D.get_truth(df, "val", threshold=3)
truth_r3, truth_r3_stats = D.restrict_truth_to_catalog(raw_truth_r3, item_pos)
users_r3 = list(truth_r3.keys())
print(f"rating>=3 validation cohort: {truth_r3_stats['users_after']} users "
      f"(excluded not-in-catalog: {truth_r3_stats['excluded_share']:.2%})")

rows_r3 = []
for name, model in final_models.items():
    rec = E.generate_rec_lists(model, users_r3, seen, catalog_items, item_pos, K)
    m = M.evaluate_recommendations(rec, truth_r3, K)
    rows_r3.append({"model": name, **m})

pd.DataFrame(rows_r3).set_index("model")[["Recall@K", "Precision@K", "MAP@K", "NDCG@K"]] \
    .sort_values("NDCG@K", ascending=False).round(4)

rating>=3 validation cohort: 6019 users (excluded not-in-catalog: 0.01%)


,Recall@K,Precision@K,MAP@K,NDCG@K
model,,,,
Item-kNN,0.0813,0.1048,0.0607,0.1239
Hybrid,0.0837,0.1040,0.0596,0.1233
ALS,0.0792,0.0884,0.0487,0.1059
BPR,0.0665,0.0690,0.0381,0.0859
Popularity,0.0445,0.0756,0.0402,0.0856
Content-Based,0.0158,0.0196,0.0089,0.0230


The model ranking under `rating >= 3` matches the `rating >= 4` ranking from
Sections 4.2–4.6 (Item-kNN and Hybrid on top, Content-Based last) — the
relevance threshold changes the absolute numbers (a looser threshold makes
every model look better, since more items count as "hits") but not which
model wins on validation, so `rating >= 4` is kept as the relevance
definition for the final test evaluation.